# ប្រព័ន្ធ POS Django — ផ្នែក 2: Forms, Authentication

## មាតិកា (Table of Contents)
1. [លទ្ធផលការសិក្សា](#learning-outcomes)
2. [រំលឹកមេរៀន: តើយើងបានធ្វើអ្វីខ្លះហើយពីផ្នែក 1](#recap)
3. [Django Forms — បង្កើត Orders តាមរយៈគេហទំព័រ](#django-forms)
4. [User Authentication — Staff Login & Logout](#user-authentication)
5. [Protecting Views with `@login_required`](#protecting-views)
6. [ភ្ចាប់ Orders ជាមួយ Logged-In Cashier](#tying-orders)
7. [កិច្ចការផ្ទះ](#homework)

> **គោលបំណងនៃផ្នែក 2**: ពង្រីកមូលដ្ឋាន POS ពី​ផ្នែក 1 — បន្ថែមសំណុំបែបបទតាមវេបសម្រាប់ cashier ដើម្បីបញ្ចូល order, អោយបុគ្គលិក login មុនពេលប្រើប្រព័ន្ធ។


---

## លទ្ធផលការសិក្សា (Learning Outcomes)

នៅចុងបញ្ចប់មេរៀននេះ អ្នកអាចធ្វើបាន៖

| # | លទ្ធផល |
|---|---|
| 1 | **ចេះបង្កើត Django Forms** — បង្កើត `ModelForm` ជា class ដើម្បីទទួល និងផ្ទៀងផ្ទាត់ទិន្នន័យពី browser |
| 2 | **ដោះស្រាយ POST Requests** — សរសេរ view ដែលដំណើរការ form submission និងរក្សាទុកទៅ database យ៉ាងសុវត្ថិភាព |
| 3 | **អនុវត្ត Authentication** — ប្រើប្រព័ន្ធ login/logout និង password ដែល Django ផ្តល់ឱ្យ |
| 4 | **ការពារ Views** — កំណត់ឲ្យទំព័រនីមួយៗសម្រាប់បុគ្គលិកដែល login តែប៉ុណ្ណោះ ដោយប្រើ `@login_required` |
| 5 | **ភ្ជាប់ទិន្នន័យទៅឲ្យ User** — ជំនួស `cashier` ជា plain-text ដោយ ForeignKey ទៅ `User` |

### ជំនាញដែលអ្នកនឹងអនុវត្ត
- ការធ្វើការជាមួយ CSRF protection របស់ Django
(ការពារការវាយប្រហារ form ពីគេហទំព័រផ្សេង — cross-site request forgery)
- ការប្រើប្រាស់ request.user
ដើម្បីស្គាល់ថា អ្នកប្រើណាកំពុង login នៅក្នុងប្រព័ន្ធ
- ការសរសេរ validation សម្រាប់ form ដោយប្រើ
is_valid() និង cleaned_data
ដើម្បីត្រួតពិនិត្យ និងយកទិន្នន័យដែលបានសម្អាតរួច
- ការសិក្សា និងអាន app built-in របស់ Django
django.contrib.auth
(ប្រព័ន្ធគ្រប់គ្រង user, login, logout, permissions)

---


---

## សង្ខេប: អ្វីដែលយើងបានបង្កើតនៅ ផ្នែក 1

នៅផ្នែក 1 យើងបានរៀបចំ skeleton នៃប្រព័ន្ធ POS៖



**អ្វីខ្វះនៅផ្នែក 1?**
- Staff អាចបង្កើត orders តែតាម admin panel តែប៉ុណ្ណោះ — មិនមាន web page ធម្មតា
- អ្នកប្រើប្រាស់ទូទៅណាក៏អាចចូល `/sales/orders/` បាន — មិនមានការទាមទារ login
- `cashier` field គឺជា free text មិនភ្ជាប់ទៅ account ពិតប្រាកដ



**ផ្នែក 2 យើងហ្នឹងដោះស្រាយបញ្ហាខាងលើ។**

---


---

## Django Forms — បង្កើត Orders តាម Web

### Django Form ជាអ្វី?

**Django Form** គឺជា class Python ដែល៖
- កំណត់ថា field អ្វីខ្លះត្រូវបង្ហាញក្នុង HTML <form>
- ពិនិត្យភាពត្រឹមត្រូវនៃទិន្នន័យ (validate) ដែល user បញ្ចូល
- បម្លែងទិន្នន័យពី POST (raw data) ទៅជា Python object ដែលបានសម្អាត (clean)

Django Form ជាអ្នកត្រួតពិនិត្យ form
- បង្ហាញ input fields
- ពិនិត្យថាត្រឹមត្រូវឬអត់
- រៀបចំ data ឲ្យស្អាត(clean) មុនយកទៅប្រើ



### ModelForm គឺជាអ្វី?

ModelForm គឺជា shortcut ដែលឆ្លាត វាបង្កើត form ដោយផ្ទាល់ពី Model នោះមានន័យថា៖
- មិនចាំបាច់សរសេរ field ដោយដៃទេ Django យក structure ពី model មកបង្កើត form ឲ្យស្រេច


**ជំនួសឱ្យការបង្ហាញទំព័រ form ដាច់ដោយឡែកសម្រាប់ “open order” ពេលចុច ＋ New Order ប្រព័ន្ធនឹងបង្កើត order ថ្មី (ស្ថានភាព open) ភ្លាមៗ ហើយនាំអ្នកគិតលុយចូលទៅកាន់អេក្រង់បញ្ចូលទំនិញ (item-entry screen) តែម្តង។**

### ដំណាក់កាល 1 — បង្កើត `sales/forms.py`

យើងត្រូវការ `OrderItemForm` តែមួយ — `OrderForm` មិនចាំបាច់ ព្រោះ order ត្រូវបានបង្កើតដោយស្វ័យប្រវត្តិ៖


In [ ]:
# sales/forms.py

from django import forms
from .models import OrderItem


class OrderItemForm(forms.ModelForm):
    """One line item to add to an order."""
    class Meta:
        model  = OrderItem
        fields = ['product', 'quantity']

    def clean_quantity(self):
        """Validation: quantity must be at least 1."""
        qty = self.cleaned_data['quantity']
        if qty < 1:
            raise forms.ValidationError("Quantity must be at least 1.")
        return qty


### គំនិតសំខាន់ៗសម្រាប់ Form 

| Concept | តើវាធ្វើអ្វី |
|---|---|
| `ModelForm` | បង្កើត field ពី Model ដោយស្វ័យប្រវត្តិ — មិនមានស្ទួន |
| `fields = [...]` | បង្ហាញតែ field នៅក្នុង [...] ក្នុង form តែប៉ុណ្ណោះ |
| `form.is_valid()` | ដំណើរការ validation ទាំងអស់; ត្រឡប់(return) True បើទិន្នន័យត្រឹមត្រូវ |
| `form.cleaned_data` | dictionary របស់តម្លៃ Python ដែលបាន validated |
| `clean_<field>()` | បន្ថែម logic validation ផ្ទាល់ខ្លួនសម្រាប់ field ជាក់លាក់មួយ |
| `form.save()` | បង្កើត ឬ update object នៅក្នុង database |
| `commit=False` | បង្កើត object មុនសិន ដោយមិនទាន់ save — អាចកំណត់ field បន្ថែមសិន |


### ដំណាក់កាល 2 — ធ្វើបច្ចុប្បន្នភាព `views.py` ដើម្បីដំណើរការ Form

មាន Views ២ សម្រាប់ដំណើរការ Order (Order Workflow)៖
- `create_order` — បង្កើត order ថ្មី (ស្ថានភាព open) ភ្លាមៗ ហើយបញ្ជូន (redirect) ទៅទំព័របន្ទាប់ ដោយមិនបង្ហាញ form
- `add_item` — អនុញ្ញាតឲ្យអ្នកគិតលុយ បន្ថែមទំនិញ (product lines) ចូលក្នុង order បន្ទាប់មកអាចកំណត់ថា order បានបង់ប្រាក់រួច (paid)

In [ ]:
# sales/views.py  (add these imports and views at the bottom of the file)

from django.shortcuts import render, get_object_or_404, redirect
from django.contrib.auth.decorators import login_required
from .models import Product, Order, OrderItem
from .forms import OrderItemForm


# ── existing views (product_list, product_detail, order_list) stay above ──


@login_required
def create_order(request):
    """Instantly create an open order and jump straight to the add-items page."""
    order = Order.objects.create(
        cashier=request.user,
        status='open',
    )
    return redirect('add_item', pk=order.pk)


@login_required
def add_item(request, pk):
    """
    Let the cashier add line items to an open order, then mark it paid.
    """
    order = get_object_or_404(Order, pk=pk)

    if request.method == 'POST':
        # "Mark as Paid" button
        if 'mark_paid' in request.POST:
            order.status = 'paid'
            order.save()
            return redirect('order_list')

        # Add a line item
        item_form = OrderItemForm(request.POST)
        if item_form.is_valid():
            item = item_form.save(commit=False)
            item.order      = order
            item.unit_price = item.product.price   # snapshot the current price
            item.save()
            return redirect('add_item', pk=order.pk)
    else:
        item_form = OrderItemForm()

    return render(request, 'sales/add_item.html', {
        'order':     order,
        'item_form': item_form,
        'items':     order.items.select_related('product'),
    })


### ដំណាក់កាល 3 — បង្កើត Template សម្រាប់ Add Item

តែ **មួយ** template គែមគគ្រប់ប្រតិបត្តិការ order workflow ឥឡូវនេះ។ បង្កើត `sales/templates/sales/add_item.html`:


In [ ]:
<!-- sales/templates/sales/add_item.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Order #{{ order.pk }}</title>
    <style>
        body  { font-family: Arial, sans-serif; max-width: 860px; margin: 40px auto; background: #f0f4f8; padding: 0 20px; }
        table { width: 100%; border-collapse: collapse; background: white; border-radius: 10px;
                overflow: hidden; box-shadow: 0 2px 6px rgba(0,0,0,0.08); margin-bottom: 24px; }
        th { background: #2b6cb0; color: white; padding: 10px 14px; text-align: left; }
        td { padding: 10px 14px; border-bottom: 1px solid #e2e8f0; }
        .total { font-size: 1.3em; font-weight: bold; color: #276749; }
        form.add  { background: white; padding: 20px 24px; border-radius: 10px;
                    box-shadow: 0 2px 6px rgba(0,0,0,0.08); display: flex; gap: 12px; align-items: flex-end; }
        form.add label { font-weight: bold; display: block; margin-bottom: 4px; }
        form.add select, form.add input { padding: 8px 10px; border: 1px solid #cbd5e0; border-radius: 6px; font-size: 1em; }
        .btn-add  { background: #2b6cb0; color: white; border: none; border-radius: 6px;
                    padding: 9px 22px; font-size: 1em; cursor: pointer; }
        .btn-paid { background: #276749; color: white; border: none; border-radius: 6px;
                    padding: 10px 28px; font-size: 1.05em; cursor: pointer; margin-top: 16px; }
    </style>
</head>
<body>
    <h1>Order #{{ order.pk }} — Adding Items</h1>
    <p>Cashier: <strong>{{ order.cashier }}</strong></p>

    <!-- Current items table -->
    <table>
        <thead><tr><th>Product</th><th>Qty</th><th>Unit Price</th><th>Subtotal</th></tr></thead>
        <tbody>
        {% for item in items %}
        <tr>
            <td>{{ item.product.name }}</td>
            <td>{{ item.quantity }}</td>
            <td>${{ item.unit_price }}</td>
            <td>${{ item.subtotal }}</td>
        </tr>
        {% empty %}
        <tr><td colspan="4" style="color:#888;text-align:center">No items yet.</td></tr>
        {% endfor %}
        </tbody>
    </table>
    <p class="total">Order Total: ${{ order.total }}</p>

    <!-- Add item form -->
    <form class="add" method="post">
        {% csrf_token %}
        {% for field in item_form %}
        <div>
            <label>{{ field.label }}</label>
            {{ field }}
        </div>
        {% endfor %}
        <button class="btn-add" type="submit">＋ Add Item</button>
    </form>

    <!-- Mark Paid -->
    <form method="post">
        {% csrf_token %}
        <button class="btn-paid" name="mark_paid" type="submit">✔ Mark Order as Paid</button>
    </form>
</body>
</html>


### ដំណាក់កាល 4 — បន្ថែម URL Patterns ថ្មី

ធ្វើឲ្យ `sales/urls.py` រួមបញ្ចូល views ថ្មីទាំងពីរ៖


In [ ]:
# sales/urls.py  (updated — replace the whole file)

from django.urls import path
from . import views

urlpatterns = [
    # ── Part 1 views ──────────────────────────────────────
    path('products/',              views.product_list,   name='product_list'),
    path('products/<int:pk>/',     views.product_detail, name='product_detail'),
    path('orders/',                views.order_list,     name='order_list'),

    # ── Part 2 views (new) ────────────────────────────────
    path('orders/new/',            views.create_order,   name='create_order'),
    path('orders/<int:pk>/items/', views.add_item,       name='add_item'),
]

# Full URL examples:
# GET  /sales/orders/new/         → blank order form
# POST /sales/orders/new/         → creates order, redirects to add-items page
# GET  /sales/orders/3/items/     → add items to order #3
# POST /sales/orders/3/items/     → save a line item OR mark order as paid


---

## Authentication អ្នកប្រើ — Staff Login & Logout

Django មានប្រព័ន្ធ authentication ល្អរួមចំណែក — អ្នកមិនចាំបាច់សរសេរ login pages ពីសូន្យទេ។

### វា​ដំណើរការ​យ៉ាងដូចម្តេច


### ដំណាក់កាលទី 1 — Wire Up Auth URLs និង Default Home Page ក្នុង `posdb/urls.py`

- `RedirectView` — បញ្ជូនអ្នកប្រើប្រាស់ដែល login `/` ទៅកាន់ login page
- `django.contrib.auth.urls` — ផ្តល់ login, logout, និង password views


In [ ]:
# posdb/urls.py  (updated)

from django.contrib import admin
from django.urls import path, include
from django.views.generic import RedirectView

urlpatterns = [
    path('',          RedirectView.as_view(url='/accounts/login/'), name='home'),  # / → login page"
    path('admin/',    admin.site.urls),
    path('sales/',    include('sales.urls')),
    path('accounts/', include('django.contrib.auth.urls')),  # login, logout, password change
]

# URL map:
# /                  → redirect to /accounts/login/
# /accounts/login/   → login page
# /accounts/logout/  → logs the user out
# /sales/products/   → product catalogue  (requires login)
# /sales/orders/     → orders list        (requires login)
# /admin/            → Django admin panel


### ដំណាក់កាលទី 2 — បង្កើត Login Template

Django auth views នឹងស្វែងរក template មានឈ្មោះ `registration/login.html`។

បង្កើត folder `templates/registration/` នៅ **root project** និងបន្ថែម `login.html`:



> សូមធានាថា `posdb/settings.py` មាន `TEMPLATES` ដែលចាត់ទុក `BASE_DIR / 'templates'` ក្នុង `'DIRS'`:


In [ ]:
TEMPLATES = [
    {
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': [BASE_DIR / 'templates'],
        'APP_DIRS': True,
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
]

In [ ]:
<!-- templates/registration/login.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Staff Login — POS</title>
    <style>
        body  { font-family: Arial, sans-serif; display: flex; justify-content: center;
                align-items: center; min-height: 100vh; background: #edf2f7; margin: 0; }
        .box  { background: white; padding: 40px 48px; border-radius: 12px;
                box-shadow: 0 4px 16px rgba(0,0,0,0.1); width: 340px; }
        h1    { color: #1a202c; margin-bottom: 24px; text-align: center; }
        label { font-weight: bold; display: block; margin-top: 14px; color: #2d3748; }
        input { width: 100%; padding: 9px 11px; margin-top: 6px; border: 1px solid #cbd5e0;
                border-radius: 6px; box-sizing: border-box; font-size: 1em; }
        button { margin-top: 24px; width: 100%; padding: 11px; background: #2b6cb0;
                 color: white; border: none; border-radius: 7px; font-size: 1.05em; cursor: pointer; }
        button:hover { background: #2c5282; }
        .errors { color: #c53030; font-size: 0.88em; margin-top: 10px; }
    </style>
</head>
<body>
    <div class="box">
        <h1>🛒 POS Staff Login</h1>
        <form method="post">
            {% csrf_token %}
            {{ form.as_p }}
            {% if form.errors %}
            <p class="errors">Invalid username or password. Please try again.</p>
            {% endif %}
            <button type="submit">Log In</button>
        </form>
    </div>
</body>
</html>


### ដំណាក់កាល 3 — កំណត់ Redirect URLs ក្នុង `settings.py`

បន្ទាប់ពី login/logout, Django នឹង redirect user ទៅទីកន្លែងមួយ។ ប្រាប់វាថាត្រូវ redirect ទៅណា៖


In [ ]:
#-----

ALLOWED_HOSTS = ['127.0.0.1', 'localhost']

#-----

# posdb/settings.py  — add these two lines anywhere at the bottom

LOGIN_REDIRECT_URL  = '/sales/products/'   # after login → product catalogue
LOGOUT_REDIRECT_URL = '/accounts/login/'   # after logout → back to login page

# With these settings + the root RedirectView, the full navigation cycle is:
#   /  →  /accounts/login/  →  (log in)  →  /sales/products/
#   (log out)  →  /accounts/login/


## ការពារ Views ដោយ `@login_required`

បន្ថែម `@login_required` លើ view function នីមួយៗ នឹង **រារាំងអ្នកមិនបាន authenticate**។ ប្រសិនបើអ្នកមិនបាន login ពេលព្យាយាមចូល URL នោះ Django នឹង redirect ទៅ `/accounts/login/`។


In [ ]:
# sales/views.py  — complete file with authentication applied to all views

from django.shortcuts import render, get_object_or_404, redirect
from django.contrib.auth.decorators import login_required
from .models import Product, Order, OrderItem
from .forms import OrderItemForm


@login_required
def product_list(request):
    products = Product.objects.filter(is_active=True)
    return render(request, 'sales/product_list.html', {'products': products})


@login_required
def product_detail(request, pk):
    product = get_object_or_404(Product, pk=pk)
    return render(request, 'sales/product_detail.html', {'product': product})


@login_required
def order_list(request):
    orders = Order.objects.all()
    return render(request, 'sales/order_list.html', {'orders': orders})


@login_required
def create_order(request):
    """Instantly create an open order and jump straight to the add-items page."""
    order = Order.objects.create(
        cashier=request.user,
        status='open',
    )
    return redirect('add_item', pk=order.pk)


@login_required
def add_item(request, pk):
    order = get_object_or_404(Order, pk=pk)
    if request.method == 'POST':
        if 'mark_paid' in request.POST:
            order.status = 'paid'
            order.save()
            return redirect('order_list')
        item_form = OrderItemForm(request.POST)
        if item_form.is_valid():
            item            = item_form.save(commit=False)
            item.order      = order
            item.unit_price = item.product.price
            item.save()
            return redirect('add_item', pk=order.pk)
    else:
        item_form = OrderItemForm()
    return render(request, 'sales/add_item.html', {
        'order': order,
        'item_form': item_form,
        'items': order.items.select_related('product'),
    })


### បន្ថែម Link ចេញពី session (Logout) ទៅក្នុង Templates របស់អ្នក

គ្រប់ទំព័រត្រូវមានវិធីសម្រាប់ staff ចេញពីប្រព័ន្ធ។ បន្ថែម snippet នេះទៅ navigation bar ក្នុង `product_list.html` និង `order_list.html`:


In [ ]:
<!-- Add this <nav> block to the top of product_list.html and order_list.html -->
<nav>
    <a href="/sales/products/">Products</a>
    <a href="/sales/orders/">Orders</a>
    <a href="/sales/orders/new/">＋ New Order</a>
    <a href="/admin/">Admin</a>
    <span style="float:right">
        👤 {{ request.user.username }}
        <form method="post" action="/accounts/logout/" style="display:inline">
            {% csrf_token %}
            <button type="submit" style="background:none;border:none;color:#2b6cb0;cursor:pointer;padding:0;font-size:1em">
                Log Out
            </button>
        </form>
    </span>
</nav>


---

## ភ្ជាប់ Orders ទៅ Cashier ដែលបាន login

បច្ចុប្បន្ន `cashier` ជា `CharField` តែប៉ុណ្ណោះ។ វិធីល្អជាងគេគឺប្រើ **ForeignKey ទៅ model `User`** ដូច្នេះ order នីមួយៗភ្ជាប់ទៅ account staff ពិតប្រាកដ។

### កែប្រែ `Order` Model

​*** មុនហ្នឹងធ្វើការកែប្រែ សូមធ្វើការលុប `order` ពីមុនទាំងអស់សិន ***


In [ ]:
# sales/models.py  — update the Order model

from django.db import models
from django.contrib.auth.models import User   # ← import Django's built-in User


class Order(models.Model):
    STATUS_CHOICES = [
        ('open',      'Open'),
        ('paid',      'Paid'),
        ('refunded',  'Refunded'),
        ('cancelled', 'Cancelled'),
    ]

    # Replace 'cashier = models.CharField(...)' with a ForeignKey:
    cashier    = models.ForeignKey(
                    User,
                    on_delete=models.SET_NULL,   # keep the order if the staff account is deleted
                    null=True,
                    related_name='orders',
                 )
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        return sum(item.subtotal for item in self.items.all())

    def __str__(self):
        name = self.cashier.username if self.cashier else 'unknown'
        return f"Order #{self.pk}  [{self.status.upper()}]  by {name}  —  ${self.total:.2f}"

    #class Meta:
    #    ordering = ['-created_at']


បន្ទាប់ពីផ្លាស់ប្តូរម៉ូដែល run migrations ៖

```bash
python manage.py makemigrations sales
python manage.py migrate
```

បន្ទាប់មកធ្វើឲ្យ `create_order` ក្នុង `views.py` assign `request.user` ត្រូវការ object `User` ទៅ `order.cashier` ដោយផ្ទាល់៖

```python
order.cashier = request.user   # ← assign the User object, not a string
```

### សង្ខេប Authentication

| Feature | វា​ដំណើរការ​យ៉ាងដូចម្តេច |
|---|---|
| `include('django.contrib.auth.urls')` | បន្ថែម login, logout, password-change URLs ឲ្យដោយស្វ័យ |
| `@login_required` | រារាំង anonymous users ពី view |
| `request.user` | Object `User` ដែលបាន login (ឬ `AnonymousUser`) |
| `request.user.is_authenticated` | `True` ប្រសិនបើមានអ្នក login |
| `LOGIN_REDIRECT_URL` | ទីតាំង redirect បន្ទាប់ពី login ជោគជ័យ |
| `LOGOUT_REDIRECT_URL` | ទីតាំង redirect បន្ទាប់ពី logout |
| **CSRF token** | `{% csrf_token %}` — ត្រូវនៅក្នុង POST form រាល់សំណុំ, គឺការការពារការវាយប្រហារ CSRF |

---


---

## សេចក្តីសន្និដ្ឋាន (Conclusion)

### អ្វីដែលអ្នកបានរៀននៅ ផ្នែក 2

| Concept | វាធ្វើអ្វី |
|---|---|
| **`ModelForm`** | បង្កើត form ពី Model ដោយស្វ័យ — មិនចាំបាច់ចម្លង field |
| **`is_valid()` / `cleaned_data`** | ផ្ទៀងផ្ទាត់ POST data និងបង្ហាញតម្លៃ Python ស្អាត |
| **`commit=False`**  | អនុញ្ញាតកែប្រែ instance មុនរក្សាទុក |
| **`Order.objects.create()`** | បង្កើត record បន្ទាន់ ដោយគ្មាន form — ប្រើសម្រាប់ order ខ្លួនឯង |
| **`@login_required`** | ការការពារ១បន្ទាត់ — redirect anonymous users ទៅ login page |
| **`request.user`** | Object `User` ដែលបាន login, មាននៅក្នុង view ដែលបានការពារ |
| **`ForeignKey(User)`** | ភ្ជាប់ `Order` ទៅ account staff ពិតប្រាកដ |
| **`RedirectView`** | view ដើម្បី redirect URL មួយទៅ URL មួយទៀត — ប្រើសម្រាប់ homepage |
| **`include('django.contrib.auth.urls')`** | ផ្ដល់ login/logout/password views ដោយម្តង |

### រចនាសម្ព័ន្ធឯកសារ Part 2 ពេញលេញ




### ផែនទី URL ពេញលេញ

| URL | វាធ្វើអ្វី |
|---|---|
| `/` | Redirected to `/accounts/login/` |
| `/accounts/login/` | Staff login page |
| `/accounts/logout/` | Logs out, redirects to `/accounts/login/` |
| `/sales/products/` | Product catalogue (login required) |
| `/sales/products/<id>/` | Product detail page (login required) |
| `/sales/orders/` | All orders (login required) |
| `/sales/orders/new/` | Create order + go to add-items (login required) |
| `/sales/orders/<id>/items/` | Add items / mark paid (login required) |
| `/admin/` | Django admin panel |



### Workflow នៃ Order (ផ្នែក 2)



### សប្តាហ៍ក្រោយ

- **software deployment** — using `pythonanywhere` and `git`

---


---

## ការផ្តល់ការងារផ្ទះ (Homework) 🏠

បំពេញភារកិច្ចខាងក្រោម ដើម្បីពង្រឹងចំណេះដែលបានរៀននៅ ផ្នែក 2។ រាល់ភារកិច្ចស្ថិតលើកូដពីមេរៀននេះ។

---

### ភារកិច្ច 1 — ព្យាយាម Workflow ពេញលេញ (ងាយ)

1. ប្រើ server: `python manage.py runserver`
2. ចូលទៅ `http://127.0.0.1:8000/sales/products/` — អ្នកគួរត្រូវបាន redirect ទៅ login page។
3. ចូលជាមួយ superuser account។
4. បង្កើត order ថ្មី តាម `http://127.0.0.1:8000/sales/orders/new/`
5. បន្ថែមយ៉ាងហោចណាស់ **3 line items** និង mark order ជា **Paid**។
6. ធ្វើតេស្តថា order បង្ហាញនៅ orders list មាន total ត្រឹមត្រូវ។

---



### ភារៈកិច្ច 2 — Form Validation (ងាយ–មធ្យម)

ក្នុង `sales/forms.py`, បន្ថែម validation ការផ្ទៀងផ្ទាល់ទៅ `OrderItemForm`:
- ប្រសិនបើ product ដែលជ្រើស `stock` = `0`, បោះ error `ValidationError` ជាមួយសារ `"This product is out of stock."`

```python
def clean_product(self):
        """Validation: product must be in stock."""
        product = self.cleaned_data.get('product')
        if product and product.stock == 0:
            raise forms.ValidationError("This product is out of stock.")
        return product
```

សាកល្បងដោយកំណត់ product stock = 0 ក្នុង shell និងព្យាយាមបញ្ចូលវាទៅ order។

---



### ភារកិច្ច 3 — ទំព័រ "My Orders" (មធ្យម)

បន្ថែម view ថ្មី `/sales/orders/mine/` ដែលបង្ហាញតែ orders ដែលបង្កើតដោយ user ដែល login នៅពេលនេះ៖

```python
@login_required
def my_orders(request):
    orders = Order.objects.filter(cashier=request.user)
    return render(request, 'sales/order_list.html', {'orders': orders})
```

1. បន្ថែម URL pattern ទៅ `sales/urls.py`.
2. បន្ថែម link "My Orders" ទៅ navigation bar នៅ templates របស់អ្នក.

---



### ភារកិច្ច 4 — កាត់ស្តុក (Stock Deduction) (មធ្យម–យ៉ាប់កប់)

នៅពេល `OrderItem` ត្រូវបានរក្សា (បន្ទាប់ពី `item_form.save()`), បន្ថែម logic ដើម្បីកាត់ស្តុក product ដោយ `quantity`៖

Update `add_item` view in `views.py` :
```python
if item_form.is_valid():
    item            = item_form.save(commit=False)
    item.order      = order
    item.unit_price = item.product.price
    item.save()

    # Deduct stock
    product = item.product
    product.stock -= item.quantity
    product.save()

    return redirect('add_item', pk=order.pk)
```

សាកល្បងដោយបញ្ចូល items ហើយពិនិត្យថាស្តុក product ថយចុះនៅក្នុង admin panel។

---



### ភារកិច្ច 5 — Bonus Challenge ⭐ +50pt 

បន្ថែម **Cancel Order** button នៅ `add_item.html` ដែល៖
1. កំណត់ `order.status = 'cancelled'` និង save។
2. Redirect ទៅ order list។
3. ធ្វើអោយ ឲ`ORDER` ដែល(cancelled) មិនអាចបន្ថែម items បានទៀត (ត្រួតពិនិត្យក្នុង `add_item` view និងបង្ហាញ error message ប្រសិនបើ order មិនមែន `open`)។

---


### ភារកិច្ច 6 -- Bonus Challenge ⭐ + 50pt

ប្រើប្រាស់ AI ធ្វើការ update Project អោយមើលទៅ ស្អាត ប្រទាក់ភ្នែក និង បន្ថែម Product Image